In [ ]:
import tempfile
import os
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import xarray as xr
from IPython.display import display, Markdown, HTML
import seaborn as sns
from matplotlib.ticker import FuncFormatter

from qcore import coordinates
import nn_gmm as nng
import ml_tools as mlt

logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    level=logging.WARNING
)

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.mps.is_available():
    device = "mps"

print(f"Using device: {device.upper()}")

PLOT_IMS = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

In [ ]:
result_1_dir = Path("/Users/claudy/dev/work/data/nn_gmm/results/1015_1317_full_v1_5Epochs")
result_2_dir = Path("/Users/claudy/dev/work/data/nn_gmm/results/1015_1128_full_v3_locSiteCond_20Epochs")

In [ ]:
run_config_1 = nng.RunConfig.from_yaml(result_1_dir / "run_config.yaml")
run_config_2 = nng.RunConfig.from_yaml(result_2_dir / "run_config.yaml")

In [ ]:
model_1 = torch.load(result_1_dir / "model.pt", weights_only=False, map_location=device)
model_2 = torch.load(result_2_dir / "model.pt", weights_only=False, map_location=device)

In [ ]:
site_locs = nng.loc_pre.get_random_sites(250_000)
site_locs = site_locs[site_locs[:, 1] < -36.0]
nztm_coords = coordinates.wgs_depth_to_nztm(site_locs[:, ::-1])

## $R_{Rup}$ = 25km, $M_W$ = 6.5

In [ ]:
inputs = {
    "rrup": 25,
    "rjb": 25,
    "rx": 25,
    "ry": 25,
    "magnitude": 6.5,
    "tect_type": "crustal",
    "rake": 0,
    "dip": 90,
    "dtop": 0,
    "dbottom": 5,
    "vs30": 760,
    "z1p0": 0.1,
    "z2p5": 1.0,
}
input_df = pd.DataFrame([inputs])

In [ ]:
input_df

In [ ]:
model_1_preds = nng.nn_gmm.run_predictions(model_1, run_config_1, input_df, device=device)

In [ ]:
loc_input_df = pd.DataFrame(data=site_locs, columns=["lon", "lat"])
loc_input_df["nztm_y"] = nztm_coords[:, 0]
loc_input_df["nztm_x"] = nztm_coords[:, 1]

for cur_key in inputs:
    loc_input_df[cur_key] = inputs[cur_key]

In [ ]:
model_2_preds = nng.nn_gmm.run_predictions(model_2, run_config_2, loc_input_df, device=device)

In [ ]:
model_2_pSA_median = model_2_preds.loc[:, run_config_2.pred_pSA_mean_keys].median(axis=0)
model_2_pSA_median.index = run_config_2.pSA_ims

model_2_pSA_1st = model_2_preds.loc[:, run_config_2.pred_pSA_mean_keys].quantile(0.01, axis=0)
model_2_pSA_1st.index = run_config_2.pSA_ims
model_2_pSA_16th= model_2_preds.loc[:, run_config_2.pred_pSA_mean_keys].quantile(0.16, axis=0)
model_2_pSA_16th.index = run_config_2.pSA_ims
model_2_pSA_84th= model_2_preds.loc[:, run_config_2.pred_pSA_mean_keys].quantile(0.84, axis=0)
model_2_pSA_84th.index = run_config_2.pSA_ims
model_2_pSA_99th= model_2_preds.loc[:, run_config_2.pred_pSA_mean_keys].quantile(0.99, axis=0)
model_2_pSA_99th.index = run_config_2.pSA_ims

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(run_config_1.pSA_periods, model_1_preds[run_config_1.pred_pSA_mean_keys].values.ravel(), c="blue", label="Model 1")
ax.plot(run_config_2.pSA_periods, model_2_pSA_median.values, c="red", label="Model 2")
ax.fill_between(run_config_2.pSA_periods, model_2_pSA_16th.values, model_2_pSA_84th.values, color="red", alpha=0.4, label="Model 2 16th-84th Percentile")
ax.fill_between(run_config_2.pSA_periods, model_2_pSA_1st.values, model_2_pSA_99th.values, color="red", alpha=0.1, label="Model 2 1st-99th Percentile")

ax.set_ylabel("pSA (g)")
ax.set_xlabel("Period (s)")
plt.xlim(0.01, 10)
ax.grid(
            which="both",
            linewidth=0.5,
            alpha=0.5,
            linestyle="--",
        )
ax.set_xscale("log")
ax.yaxis.set_major_formatter(FuncFormatter(nng.plot_utils.exp_formatter))
ax.legend()

fig.tight_layout()

In [ ]:
cur_im = "pSA_5.0"

res_df = pd.DataFrame(data=model_2_preds[f"{cur_im}_pred"].values -  model_1_preds[f"{cur_im}_pred"].values, columns=["residual"])
res_df[["lon", "lat"]] = model_2_preds[["lon", "lat"]]

In [ ]:
spatial_plot = nng.plots_spatial.SpatialPlot().plot_ratio(res_df, "residual", cmap_limits=(-1.0, 1.0, 2.0 / 16), continuous_cmap=True).plot_basin_boundaries(pen="0.15p,black")
spatial_plot.fig.show(dpi=900, width=1200)